# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8080 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))

1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [5]:
schema_produits = T.StructType([
    T.StructField("produit_id", T.StringType(), False),
    T.StructField("nom", T.StringType(), False),
    T.StructField("categorie", T.StringType(), False),
    T.StructField("sous_categorie", T.StringType(), False),
    T.StructField("prix", T.DoubleType(), False),
    T.StructField("poids_kg", T.DoubleType(), False)
])

schema_commandes = T.StructType([
    T.StructField("commande_id", T.StringType(), False),
    T.StructField("client_id", T.StringType(), False),
    T.StructField("horodatage", T.TimestampType(), False),
    T.StructField("evenement", T.StringType(), False),
    T.StructField("livreur_id", T.StringType(), True),
    T.StructField("detail", T.StringType(), True)
])

schema_communes = T.StructType([
    T.StructField("commune", T.StringType(), False),
    T.StructField("code_postal", T.StringType(), False),
    T.StructField("lon", T.DoubleType(), False),
    T.StructField("lat", T.DoubleType(), False)
])

schema_gps = T.StructType([
    T.StructField("timestamp", T.TimestampType(), False),
    T.StructField("livreur_id", T.StringType(), False),
    T.StructField("lon", T.DoubleType(), False),
    T.StructField("lat", T.DoubleType(), False),
    T.StructField("vitesse_kmh", T.DoubleType(), False),
    T.StructField("dans_vehicule", T.BooleanType(), False),
    T.StructField("charge_kg", T.DoubleType(), False)
])

produits = spark.read.option("header", True).schema(schema_produits).csv(f"{DATA}/produits.csv")
journal = spark.read.option("header", True).schema(schema_commandes).csv(f"{DATA}/commandes.csv")
commandes_raw = journal
communes = spark.read.option("header", True).schema(schema_communes).csv(f"{DATA}/communes.csv")
gps = spark.read.option("header", True).schema(schema_gps).csv(f"{DATA}/vehicules/*/*.csv")

print(f"produits    : {produits.count():>10,d} lignes".replace(",", " "))
print(f"commandes   : {journal.count():>10,d} lignes".replace(",", " "))
print(f"communes    : {communes.count():>10,d} lignes".replace(",", " "))
print(f"gps         : {gps.count():>10,d} lignes".replace(",", " "))


produits    :        233 lignes
commandes   :    846 535 lignes
communes    :        142 lignes


gps         :  5 174 905 lignes


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [6]:
gps = (gps.withColumn("vehicule_id", F.element_at(F.split(F.input_file_name(), "/"), -2))
          .withColumn("jour", F.to_date(F.col("timestamp"))))
gps.printSchema()
gps.show(5)

root
 |-- timestamp: timestamp (nullable = true)
 |-- livreur_id: string (nullable = true)
 |-- lon: double (nullable = true)
 |-- lat: double (nullable = true)
 |-- vitesse_kmh: double (nullable = true)
 |-- dans_vehicule: boolean (nullable = true)
 |-- charge_kg: double (nullable = true)
 |-- vehicule_id: string (nullable = true)
 |-- jour: date (nullable = true)

+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|          timestamp|livreur_id|     lon|      lat|vitesse_kmh|dans_vehicule|charge_kg|vehicule_id|      jour|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|2025-10-01 07:40:00|      L001|2.277083|48.936735|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:41:00|      L001|2.277083|48.936735|        0.0|        false|   2581.6|        V18|2025-10-01|
|2025-10-01 07:42:00|      L001|2.277084|48.936734|        0.0|        false

## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [7]:
nb_commandes = journal.select("commande_id").distinct().count()
nb_clients = journal.select("client_id").distinct().count()
nb_livreurs = journal.filter(F.col("livreur_id").isNotNull()).select("livreur_id").distinct().count()
nb_vehicules = gps.select("vehicule_id").distinct().count()

print(f"Nombre de commandes : {nb_commandes:,}".replace(",", " "))
print(f"Nombre de clients   : {nb_clients:,}".replace(",", " "))
print(f"Nombre de livreurs  : {nb_livreurs:,}".replace(",", " "))
print(f"Nombre de véhicules : {nb_vehicules:,}".replace(",", " "))


Nombre de commandes : 271 617
Nombre de clients   : 8 000
Nombre de livreurs  : 90
Nombre de véhicules : 72


2.2) Comptez les événements du journal par type.

In [8]:
journal.groupBy("evenement").count().orderBy(F.desc("count")).show()


+---------+------+
|evenement| count|
+---------+------+
|  chargee|286862|
| commande|271617|
|   livree|268682|
|    echec| 11220|
| reportee|  6960|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [9]:
repartition_chargements = (journal.groupBy("commande_id")
                           .agg(F.count(F.when(F.col("evenement") == "chargee", 1)).alias("nb_chargee"))
                           .groupBy("nb_chargee")
                           .agg(F.count("commande_id").alias("nb_commandes"))
                           .orderBy("nb_chargee"))
repartition_chargements.show()


+----------+------------+
|nb_chargee|nb_commandes|
+----------+------------+
|         0|        1637|
|         1|      254229|
|         2|       14673|
|         3|        1026|
|         4|          51|
|         5|           1|
+----------+------------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [10]:
w_cmd = Window.partitionBy("commande_id").orderBy("horodatage")
journal_prev = journal.withColumn("evenement_precedent", F.lag("evenement", 1).over(w_cmd))

(journal_prev.filter(F.col("evenement") == "chargee")
 .groupBy("evenement_precedent")
 .agg(F.count("*").alias("nb_chargee"))
 .orderBy(F.desc("nb_chargee"))
 .show())


+-------------------+----------+
|evenement_precedent|nb_chargee|
+-------------------+----------+
|           commande|    269980|
|              echec|      9950|
|           reportee|      6932|
+-------------------+----------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [11]:
cmd_statuts = (journal.groupBy("commande_id")
               .agg(F.count(F.when(F.col("evenement") == "livree", 1)).alias("nb_livree"),
                    F.count(F.when(F.col("evenement") == "retour", 1)).alias("nb_retour")))

non_livrees = cmd_statuts.filter(F.col("nb_livree") == 0)
total_non_livrees = non_livrees.count()
abandonnees = non_livrees.filter(F.col("nb_retour") > 0).count()
en_cours = non_livrees.filter(F.col("nb_retour") == 0).count()

print(f"Total commandes non livrées : {total_non_livrees:,}".replace(",", " "))
print(f"  - Abandonnées (retour)    : {abandonnees:,}".replace(",", " "))
print(f"  - Encore en cours         : {en_cours:,}".replace(",", " "))

Total commandes non livrées : 2 935
  - Abandonnées (retour)    : 1 194
  - Encore en cours         : 1 741


## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [12]:
lignes = (journal.filter(F.col("evenement") == "commande")
          .withColumn("item", F.explode(F.split(F.col("detail"), "\\|")))
          .withColumn("produit_id", F.split(F.col("item"), "x")[0])
          .withColumn("quantite", F.split(F.col("item"), "x")[1].cast(T.IntegerType()))
          .select("commande_id", "client_id", "horodatage", "produit_id", "quantite"))

print(f"Nombre de lignes de commande : {lignes.count():,}".replace(",", " "))
lignes.show(5)


Nombre de lignes de commande : 4 779 049
+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|         horodatage|produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0037|       5|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0063|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0019|       2|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0067|       1|
|   C0000001|  CL05658|2025-01-01 00:03:44|     P0069|       2|
+-----------+---------+-------------------+----------+--------+
only showing top 5 rows



3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [13]:
lignes = (lignes.join(produits, on="produit_id", how="left")
                .withColumn("montant", F.round(F.col("quantite") * F.col("prix"), 2)))
lignes.persist()

commandes = (lignes.groupBy("commande_id", "client_id", "horodatage")
             .agg(F.round(F.sum("montant"), 2).alias("montant"),
                  F.round(F.sum(F.col("quantite") * F.col("poids_kg")), 2).alias("poids_kg"),
                  F.count("produit_id").alias("nb_lignes")))
commandes.persist()

print(f"Table commandes : {commandes.count():,}".replace(",", " "))
commandes.show(5)


Table commandes : 271 617
+-----------+---------+-------------------+-------+--------+---------+
|commande_id|client_id|         horodatage|montant|poids_kg|nb_lignes|
+-----------+---------+-------------------+-------+--------+---------+
|   C0000168|  CL06333|2025-01-01 13:16:07| 209.35|   60.98|       32|
|   C0000469|  CL03079|2025-01-01 22:39:48|   80.9|   14.03|       13|
|   C0000752|  CL00230|2025-01-02 16:34:53| 218.05|   46.52|       31|
|   C0000810|  CL07244|2025-01-02 18:29:22| 239.45|    78.5|       27|
|   C0000855|  CL04907|2025-01-02 19:48:36|   70.9|   15.87|       15|
+-----------+---------+-------------------+-------+--------+---------+
only showing top 5 rows



## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [14]:
w_vj = Window.partitionBy("vehicule_id", "jour").orderBy("timestamp")
w_vj_desc = Window.partitionBy("vehicule_id", "jour").orderBy(F.desc("timestamp"))

first_pts = (gps.withColumn("rn", F.row_number().over(w_vj))
                .filter(F.col("rn") == 1)
                .select("vehicule_id", "jour", "lon", "lat"))
last_pts = (gps.withColumn("rn", F.row_number().over(w_vj_desc))
               .filter(F.col("rn") == 1)
               .select("vehicule_id", "jour", "lon", "lat"))

extremes = first_pts.union(last_pts)
positions_arrondies = (extremes.withColumn("lon_arrondi", F.round(F.col("lon"), 2))
                               .withColumn("lat_arrondi", F.round(F.col("lat"), 2))
                               .groupBy("lon_arrondi", "lat_arrondi")
                               .agg(F.count("*").alias("nb_points"))
                               .orderBy(F.desc("nb_points")))

positions_arrondies.show()
print(f"Il y a {positions_arrondies.count()} entrepôts.")


+-----------+-----------+---------+
|lon_arrondi|lat_arrondi|nb_points|
+-----------+-----------+---------+
|       2.45|      48.91|     8434|
|       2.28|      48.94|     6416|
|       2.35|      48.75|     5934|
+-----------+-----------+---------+



Il y a 3 entrepôts.


4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [15]:
clusters = (first_pts.withColumn("lon_arrondi", F.round(F.col("lon"), 2))
                     .withColumn("lat_arrondi", F.round(F.col("lat"), 2))
                     .groupBy("lon_arrondi", "lat_arrondi")
                     .agg(F.round(F.avg("lon"), 6).alias("lon"),
                          F.round(F.avg("lat"), 6).alias("lat"))
                     .orderBy("lon"))

w_rank = Window.orderBy("lon")
entrepots = (clusters.withColumn("rn", F.row_number().over(w_rank))
                     .withColumn("entrepot", F.concat(F.lit("E"), F.col("rn")))
                     .select("entrepot", "lon", "lat"))
entrepots.show()

# Attribution de chaque véhicule à son entrepôt le plus fréquent
v_first = first_pts.crossJoin(F.broadcast(entrepots.select(
    F.col("entrepot").alias("e_nom"),
    F.col("lon").alias("e_lon"),
    F.col("lat").alias("e_lat")
))).withColumn("dist", haversine("lon", "lat", "e_lon", "e_lat"))

w_closest = Window.partitionBy("vehicule_id", "jour").orderBy("dist")
v_closest = v_first.withColumn("rn", F.row_number().over(w_closest)).filter(F.col("rn") == 1)

w_veh = Window.partitionBy("vehicule_id").orderBy(F.desc("nb_jours"))
vehicule_entrepot = (v_closest.groupBy("vehicule_id", F.col("e_nom").alias("entrepot"))
                              .agg(F.count("jour").alias("nb_jours"))
                              .withColumn("rn", F.row_number().over(w_veh))
                              .filter(F.col("rn") == 1)
                              .select("vehicule_id", "entrepot")
                              .orderBy("vehicule_id"))

vehicule_entrepot.groupBy("entrepot").agg(F.count("vehicule_id").alias("nb_vehicules")).orderBy("entrepot").show()


16:06:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:36 WARN WindowExec: No Partition Defined for Window operat

+--------+--------+---------+
|entrepot|     lon|      lat|
+--------+--------+---------+
|      E1|2.277069|48.936713|
|      E2|2.351514|48.747047|
|      E3|2.448959|48.906702|
+--------+--------+---------+



16:06:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
16:06:48 WARN WindowExec: No Partition Defined for Window operat

+--------+------------+
|entrepot|nb_vehicules|
+--------+------------+
|      E1|          24|
|      E2|          24|
|      E3|          24|
+--------+------------+



## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

In [16]:
w_lj = Window.partitionBy("livreur_id", "jour").orderBy("timestamp")
gps = (gps.withColumn("prev_dans_vehicule", F.lag("dans_vehicule", 1).over(w_lj))
          .withColumn("changement", F.when(F.col("prev_dans_vehicule").isNull(), 0)
                                    .when((F.col("prev_dans_vehicule") == True) & (F.col("dans_vehicule") == False), 1)
                                    .otherwise(0)))

(gps.filter((F.col("livreur_id") == "L001") & 
            (F.col("jour") == "2025-01-06") & 
            (F.col("changement") == 1))
    .select("timestamp", "livreur_id", "lon", "lat", "vitesse_kmh", "dans_vehicule", "changement")
    .show(10))


+-------------------+----------+--------+---------+-----------+-------------+----------+
|          timestamp|livreur_id|     lon|      lat|vitesse_kmh|dans_vehicule|changement|
+-------------------+----------+--------+---------+-----------+-------------+----------+
|2025-01-06 09:54:00|      L001|2.311953|48.871389|        0.0|        false|         1|
|2025-01-06 10:00:00|      L001|2.313447|48.870848|        0.0|        false|         1|
|2025-01-06 10:09:00|      L001|2.316012|48.872919|        0.0|        false|         1|
|2025-01-06 10:23:00|      L001|2.325412|48.874465|        0.0|        false|         1|
|2025-01-06 10:36:00|      L001| 2.32844|48.866695|        0.0|        false|         1|
|2025-01-06 10:54:00|      L001|2.337183|48.866435|        0.0|        false|         1|
|2025-01-06 11:12:00|      L001|2.317523|48.857452|        0.0|        false|         1|
|2025-01-06 11:38:00|      L001|2.317717|48.850169|        0.0|        false|         1|
|2025-01-06 12:02:00|

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

In [17]:
gps = gps.withColumn("numero_livraison", F.sum("changement").over(w_lj))

(gps.filter((F.col("livreur_id") == "L001") & 
            (F.col("jour") == "2025-01-06") & 
            (F.col("dans_vehicule") == False))
    .groupBy("numero_livraison")
    .agg(F.min("timestamp").alias("premier_point"),
         F.max("timestamp").alias("dernier_point"),
         F.count("*").alias("minutes_a_pied"))
    .orderBy("numero_livraison")
    .show(15))


+----------------+-------------------+-------------------+--------------+
|numero_livraison|      premier_point|      dernier_point|minutes_a_pied|
+----------------+-------------------+-------------------+--------------+
|               0|2025-01-06 07:43:00|2025-01-06 08:26:00|            44|
|               1|2025-01-06 09:54:00|2025-01-06 09:57:00|             4|
|               2|2025-01-06 10:00:00|2025-01-06 10:06:00|             7|
|               3|2025-01-06 10:09:00|2025-01-06 10:18:00|            10|
|               4|2025-01-06 10:23:00|2025-01-06 10:26:00|             4|
|               5|2025-01-06 10:36:00|2025-01-06 10:43:00|             8|
|               6|2025-01-06 10:54:00|2025-01-06 10:57:00|             4|
|               7|2025-01-06 11:12:00|2025-01-06 11:29:00|            18|
|               8|2025-01-06 11:38:00|2025-01-06 11:54:00|            17|
|               9|2025-01-06 12:02:00|2025-01-06 12:05:00|             4|
|              10|2025-01-06 12:15:00|

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

In [18]:
livraisons = (gps.filter(F.col("dans_vehicule") == False)
                 .groupBy("livreur_id", "jour", "numero_livraison")
                 .agg(F.first("vehicule_id").alias("vehicule_id"),
                      F.min("timestamp").alias("debut"),
                      F.max("timestamp").alias("fin"),
                      F.count("*").alias("duree_min"),
                      F.round(F.avg("lon"), 6).alias("lon"),
                      F.round(F.avg("lat"), 6).alias("lat")))
livraisons.persist()

print(f"Table livraisons : {livraisons.count():,} lignes".replace(",", " "))
livraisons.show(5)


Table livraisons : 273 593 lignes
+----------+----------+----------------+-----------+-------------------+-------------------+---------+--------+---------+
|livreur_id|      jour|numero_livraison|vehicule_id|              debut|                fin|duree_min|     lon|      lat|
+----------+----------+----------------+-----------+-------------------+-------------------+---------+--------+---------+
|      L002|2025-05-09|               0|        V03|2025-05-09 07:41:00|2025-05-09 08:23:00|       43|2.277056|48.936734|
|      L002|2025-05-09|               1|        V03|2025-05-09 10:37:00|2025-05-09 10:41:00|        5|2.316069|48.842647|
|      L002|2025-05-09|               2|        V03|2025-05-09 10:49:00|2025-05-09 10:57:00|        9|2.312651|48.842491|
|      L002|2025-05-09|               3|        V03|2025-05-09 11:01:00|2025-05-09 11:09:00|        9| 2.30845| 48.84163|
|      L002|2025-05-09|               4|        V03|2025-05-09 11:13:00|2025-05-09 11:17:00|        5|2.307956|4

5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

In [19]:
events_liv = (journal.filter(F.col("evenement").isin("livree", "echec"))
                     .withColumn("jour", F.to_date(F.col("horodatage")))
                     .withColumnRenamed("horodatage", "signature"))

livraisons_ev = (livraisons.join(
    events_liv,
    on=(livraisons.livreur_id == events_liv.livreur_id) &
       (livraisons.jour == events_liv.jour) &
       (events_liv.signature >= livraisons.debut - F.expr("INTERVAL 1 MINUTE")) &
       (events_liv.signature <= livraisons.fin + F.expr("INTERVAL 1 MINUTE")),
    how="left"
).select(
    livraisons["livreur_id"],
    livraisons["jour"],
    livraisons["numero_livraison"],
    livraisons["vehicule_id"],
    livraisons["debut"],
    livraisons["fin"],
    livraisons["duree_min"],
    livraisons["lon"],
    livraisons["lat"],
    events_liv["signature"],
    events_liv["evenement"],
    events_liv["commande_id"],
    events_liv["client_id"]
))
livraisons_ev.persist()

print(f"Table livraisons_ev : {livraisons_ev.count():,} lignes".replace(",", " "))
livraisons_ev.show(5)


Table livraisons_ev : 307 237 lignes
+----------+----------+----------------+-----------+-------------------+-------------------+---------+--------+---------+-------------------+---------+-----------+---------+
|livreur_id|      jour|numero_livraison|vehicule_id|              debut|                fin|duree_min|     lon|      lat|          signature|evenement|commande_id|client_id|
+----------+----------+----------------+-----------+-------------------+-------------------+---------+--------+---------+-------------------+---------+-----------+---------+
|      L002|2025-05-09|               0|        V03|2025-05-09 07:41:00|2025-05-09 08:23:00|       43|2.277056|48.936734|               NULL|     NULL|       NULL|     NULL|
|      L002|2025-05-09|               1|        V03|2025-05-09 10:37:00|2025-05-09 10:41:00|        5|2.316069|48.842647|2025-05-09 10:40:08|   livree|   C0072365|  CL01156|
|      L002|2025-05-09|               2|        V03|2025-05-09 10:49:00|2025-05-09 10:57:00| 

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

In [20]:
total_events = events_liv.count()
events_rattaches = livraisons_ev.filter(F.col("signature").isNotNull()).select("commande_id", "signature").distinct().count()
non_rattaches = total_events - events_rattaches

print(f"Total événements (livree + echec) : {total_events:,}".replace(",", " "))
print(f"Événements rattachés              : {events_rattaches:,}".replace(",", " "))
print(f"Événements non rattachés          : {non_rattaches}")

Total événements (livree + echec) : 279 902
Événements rattachés              : 279 902
Événements non rattachés          : 0


5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

In [21]:
livraisons_ev_types = (livraisons_ev.groupBy("livreur_id", "jour", "numero_livraison")
                                   .agg(F.first("duree_min").alias("duree_min"),
                                        F.first("debut").alias("debut"),
                                        F.first("lon").alias("lon"),
                                        F.first("lat").alias("lat"),
                                        F.count("signature").alias("nb_ev")))

liv_dist_ent = livraisons_ev_types.crossJoin(F.broadcast(entrepots.select(
    F.col("lon").alias("e_lon"),
    F.col("lat").alias("e_lat")
))).withColumn("dist_m", haversine("lon", "lat", "e_lon", "e_lat") * 1000)

w_closest_ent = Window.partitionBy("livreur_id", "jour", "numero_livraison").orderBy("dist_m")
liv_type = (liv_dist_ent.withColumn("rn", F.row_number().over(w_closest_ent))
                        .filter(F.col("rn") == 1)
                        .withColumn("type",
                            F.when(F.col("nb_ev") > 0, "livraison")
                             .when(F.col("dist_m") < 300, "entrepot")
                             .otherwise("pause"))
                        .select("livreur_id", "jour", "numero_livraison", "type"))

livraisons_ev = livraisons_ev.join(liv_type, on=["livreur_id", "jour", "numero_livraison"], how="left")
livraisons_ev.persist()

(livraisons.join(liv_type, on=["livreur_id", "jour", "numero_livraison"], how="left")
           .withColumn("heure_debut", F.hour("debut") + F.minute("debut") / 60.0)
           .groupBy("type")
           .agg(F.count("*").alias("nb_livraisons"),
                F.round(F.avg("duree_min"), 2).alias("duree_moyenne_min"),
                F.round(F.avg("heure_debut"), 2).alias("heure_moyenne_debut"))
           .orderBy("type")
           .show())


+---------+-------------+-----------------+-------------------+
|     type|nb_livraisons|duree_moyenne_min|heure_moyenne_debut|
+---------+-------------+-----------------+-------------------+
| entrepot|        22368|            27.21|              11.68|
|livraison|       246258|             8.32|              12.19|
|    pause|         4967|             37.5|              12.35|
+---------+-------------+-----------------+-------------------+



5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

In [22]:
liv_ev_counts = (livraisons_ev.filter(F.col("type") == "livraison")
                              .groupBy("livreur_id", "jour", "numero_livraison")
                              .agg(F.count("signature").alias("nb_evenements"),
                                   F.countDistinct("client_id").alias("nb_clients")))

print("Répartition du nombre d'événements par livraison :")
liv_ev_counts.groupBy("nb_evenements").agg(F.count("*").alias("nb_livraisons")).orderBy("nb_evenements").show()

multi_clients = liv_ev_counts.filter(F.col("nb_clients") >= 2).count()
print(f"Nombre de livraisons concernant au moins 2 clients différents : {multi_clients:,}".replace(",", " "))


Répartition du nombre d'événements par livraison :


+-------------+-------------+
|nb_evenements|nb_livraisons|
+-------------+-------------+
|            1|       217538|
|            2|        24873|
|            3|         3053|
|            4|          592|
|            5|          155|
|            6|           29|
|            7|            8|
|            8|            5|
|            9|            4|
|           10|            1|
+-------------+-------------+



Nombre de livraisons concernant au moins 2 clients différents : 12 536


5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

In [23]:
w_ev = Window.partitionBy("livreur_id", "jour", "numero_livraison").orderBy("signature")
liv_ev_with_lag = (livraisons_ev.filter(F.col("type") == "livraison")
                                .withColumn("prev_sig", F.lag("signature", 1).over(w_ev))
                                .withColumn("duree_ev_min",
                                    F.when(F.col("prev_sig").isNull(),
                                           (F.unix_timestamp("signature") - F.unix_timestamp("debut")) / 60.0)
                                     .otherwise((F.unix_timestamp("signature") - F.unix_timestamp("prev_sig")) / 60.0)))

(liv_ev_with_lag.groupBy("evenement")
                .agg(F.round(F.avg("duree_ev_min"), 2).alias("duree_moyenne_min"),
                     F.count("*").alias("nb_evenements"))
                .show())


+---------+-----------------+-------------+
|evenement|duree_moyenne_min|nb_evenements|
+---------+-----------------+-------------+
|    echec|             2.02|        11220|
|   livree|             4.24|       268682|
+---------+-----------------+-------------+



5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

In [24]:
livreur_stats = (livraisons_ev.groupBy("livreur_id")
                              .agg(F.count(F.when(F.col("evenement") == "livree", 1)).alias("nb_signatures"),
                                   F.count(F.when(F.col("evenement") == "echec", 1)).alias("nb_echecs"),
                                   F.countDistinct("jour").alias("nb_jours_travailles"))
                              .withColumn("signatures_par_jour",
                                  F.round(F.col("nb_signatures") / F.col("nb_jours_travailles"), 2))
                              .orderBy(F.desc("signatures_par_jour")))

livreur_stats.show(10)

+----------+-------------+---------+-------------------+-------------------+
|livreur_id|nb_signatures|nb_echecs|nb_jours_travailles|signatures_par_jour|
+----------+-------------+---------+-------------------+-------------------+
|      L076|          468|       24|                 15|               31.2|
|      L089|          804|       42|                 26|              30.92|
|      L070|          543|       28|                 18|              30.17|
|      L086|          713|       26|                 24|              29.71|
|      L081|          615|       18|                 21|              29.29|
|      L072|          490|       21|                 17|              28.82|
|      L079|          459|       16|                 16|              28.69|
|      L085|          715|       36|                 25|               28.6|
|      L080|          457|       18|                 16|              28.56|
|      L082|          451|       19|                 16|              28.19|

## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

7.2) Panier moyen : montant moyen d'une commande.

7.3) Panier médian.

7.4) Top 10 des produits par chiffre d'affaires.

7.5) Top 10 des produits par quantité vendue.

7.6) Top 10 des clients par dépense sur l'année.

7.7) Top 10 des clients par nombre de commandes.

## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

8.4) Motifs d'échec : nombre d'échecs par motif.

8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*